# Indices spectraux et algèbre raster

**Cartomize · Version 1.0**

Durée indicative : 35 minutes.

## Objectifs

Calculer les 18 indices intégrés, contrôler une formule et comparer les effets d’un seuil et d’un voisinage.

## Apport de Cartomize

Plusieurs indices partagent la lecture des bandes ; le calcul par blocs limite la taille des tableaux intermédiaires.

Données : paysage pédagogique généré avec une graine fixe. Les cartes et mesures ne décrivent aucun site réel.

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [2]:
out = run_directory("03_indices_algebre")
d = demo(out/"data")

## 1. Choisir les indices et les unités

Les indices avec une constante additive, notamment EVI et SAVI, nécessitent une calibration correcte. Les données pédagogiques possèdent sept bandes, dont red edge. Le composite réel de Mvouti ne possède pas cette dernière bande.

In [3]:
registre = pd.DataFrame(cm.list_indices())
display(registre[['name','formula','bands']])
indices = cm.spectral_indices(d['image'], out/'indices.tif', indices=registre['name'].tolist(), workers=2, block_size=64)
with rasterio.open(indices) as src:
    assert src.count == 18
    display(pd.DataFrame({'bande':src.indexes,'indice':src.descriptions}))

,name,formula,bands
0,NDVI,(nir-red)/(nir+red),"(nir, red)"
1,EVI,g*(nir-red)/(nir+C1*red-C2*blue+L),"(nir, red, blue)"
2,EVI2,g*(nir-red)/(nir+2.4*red+L),"(nir, red)"
3,SAVI,(1+L)*(nir-red)/(nir+red+L),"(nir, red)"
4,OSAVI,(nir-red)/(nir+red+0.16),"(nir, red)"
5,MSAVI,(2*nir+1-sqrt((2*nir+1)**2-8*(nir-red)))/2,"(nir, red)"
6,GNDVI,(nir-green)/(nir+green),"(nir, green)"
7,NDRE,(nir-rededge1)/(nir+rededge1),"(nir, rededge1)"
8,NDWI,(green-nir)/(green+nir),"(green, nir)"
9,MNDWI,(green-swir1)/(green+swir1),"(green, swir1)"


,bande,indice
0,1,NDVI
1,2,EVI
2,3,EVI2
3,4,SAVI
4,5,OSAVI
5,6,MSAVI
6,7,GNDVI
7,8,NDRE
8,9,NDWI
9,10,MNDWI


## 2. Vérifier le NDVI

Une comparaison numérique avec la formule permet de contrôler le choix des bandes et la conservation du masque. Le seuil utilisé ensuite est pédagogique ; il ne définit pas une forêt.

In [4]:
ndvi = cm.raster.ndvi(d['image'], out/'ndvi.tif', red=3, nir=4)
manual = cm.calculate('(nir-red)/(nir+red)', {'nir':(d['image'],4),'red':(d['image'],3)}, out/'ndvi_formule.tif', block_size=64)
with rasterio.open(ndvi) as a, rasterio.open(manual) as b:
    assert np.allclose(a.read(1), b.read(1), equal_nan=True, atol=1e-6)
seuil = cm.calculate('where(n > 0.5, 1, 0)', {'n':ndvi}, out/'seuil.tif')
cm.register_index('TUTO_RATIO', 'nir/red', bands=('nir','red'), title='Rapport pédagogique', replace=True)
cm.spectral_indices(d['image'], out/'indice_personnalise.tif', ['TUTO_RATIO'])

Out[4]: PosixPath('tutorials/outputs/03_indices_algebre_p_2zp34a/indice_personnalise.tif')


## 3. Généraliser et résumer

Le filtre focal moyenne un voisinage spatial. La réduction combine des rasters cellule par cellule. Ici, la moyenne des deux calculs équivalents vérifie leur cohérence ; elle ne représente pas une série temporelle.

In [5]:
lisse = cm.focal(ndvi, out/'ndvi_lisse.tif', statistic='mean', size=5, block_size=64)
mean = cm.reduce_rasters([ndvi, manual], out/'ndvi_moyen.tif', statistic='mean', block_size=64)
statistics = cm.raster.zonal_stats(ndvi, cm.read_file(d['zones']))
display(statistics.drop(columns='geometry'))
fig=show_raster(ndvi, 'NDVI sur réflectances simulées', cmap='YlGn', vmin=-1, vmax=1)
gallery(fig,'03_ndvi');plt.show()

,name,group,count,min,max,mean,sum,std
0,Secteur ouest,Ensemble,4240,-0.902941,0.888582,0.395311,1676.117975,0.550678
1,Secteur est,Ensemble,4311,0.052864,0.881115,0.257800,1111.375785,0.295588


## 4. Explorer un seuil dans Jupyter

Le curseur recalcule la proportion de pixels au-dessus du seuil. GitHub affiche uniquement le résultat statique ; le contrôle fonctionne après exécution dans Jupyter.

In [6]:
import ipywidgets as widgets
with rasterio.open(ndvi) as src:
    values=src.read(1,masked=True)
@widgets.interact(seuil=widgets.FloatSlider(value=.5,min=-1,max=1,step=.05,description='Seuil'))
def proportion(seuil):
    valid=values.compressed()
    print(f'{100*np.count_nonzero(valid>seuil)/len(valid):.1f} % des pixels valides au-dessus du seuil')

40.6 % des pixels valides au-dessus du seuil


## 5. Comparer les statistiques disponibles

Le comptage retourne le nombre de valeurs valides. La médiane est une réduction multirasters ; le filtre focal de Cartomize propose une étendue (`range`), sans médiane.

In [7]:
for statistic in ('mean','sum','std','min','max','range','count'):
    cm.focal(ndvi,out/f'focal_{statistic}.tif',statistic=statistic,size=3,block_size=64)
for statistic in ('mean','sum','std','min','max','median','count'):
    cm.reduce_rasters([ndvi,manual],out/f'reduction_{statistic}.tif',statistic=statistic,block_size=64)
print('7 statistiques focales et 7 réductions exécutées.')

7 statistiques focales et 7 réductions exécutées.


## Exercice

Calculer NDMI avec la calculatrice puis comparer au résultat du registre. Pourquoi une valeur élevée de NDVI ne suffit-elle pas à identifier une forêt ?

<details>
<summary>Éléments de correction</summary>

Utiliser `(nir-swir1)/(nir+swir1)` avec les bandes 4 et 5. Le NDVI mesure un contraste spectral ; cultures, végétation herbacée et forêts peuvent partager des valeurs élevées. Une nomenclature et une validation sont nécessaires.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.